# IdiomX – Task 2 Inference Demo

This notebook provides a lightweight inference pipeline for **Task 2: Context → Idiom Retrieval**.

It loads the saved retrieval artifacts from the main benchmark notebook and applies the final **Hybrid + Reranker** pipeline to new input sentences.

## What this notebook does

- loads the idiom bank and saved embeddings
- rebuilds the lexical index
- loads the dense embedder and reranker
- predicts the most likely idioms for one or more English sentences

This notebook is designed for simple demonstration and quick testing.

In [24]:
# [B1.1] Core imports

from pathlib import Path
import warnings
import pickle
import sys
import subprocess

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Core imports loaded.")

Core imports loaded.


In [25]:
# [B1.2] Ensure required libraries are available

# sentence-transformers
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers already installed.")
except ImportError:
    print("Installing sentence-transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers installed and imported.")

# rank_bm25
try:
    from rank_bm25 import BM25Okapi
    print("rank_bm25 already installed.")
except ImportError:
    print("Installing rank_bm25...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])
    from rank_bm25 import BM25Okapi
    print("rank_bm25 installed and imported.")

sentence-transformers already installed.
rank_bm25 already installed.


### Load saved Task 2 inference artifacts

We load the saved idiom bank, dense embeddings, BM25 inputs, and index mappings
required for the final Hybrid + Reranker inference pipeline.

In [26]:
# [B1.3 - FINAL] Load and validate saved Task 2 inference artifacts

ARTIFACT_DIR = Path("../artifacts/task2")

# Check that all required files exist
required_files = [
    "idiom_bank.csv",
    "idiom_embeddings.npy",
    "idiom_to_idx.pkl",
    "idx_to_idiom.pkl",
    "bm25_tokens.pkl",
    "model_config.pkl"
]

missing_files = [
    file_name for file_name in required_files
    if not (ARTIFACT_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing artifacts: {missing_files}\n"
        f"Please run the benchmark notebook first to generate them."
    )

# -----------------------------
# Load semantic idiom bank
# -----------------------------
idiom_bank_df = pd.read_csv(
    ARTIFACT_DIR / "idiom_bank.csv",
    encoding="utf-8-sig"
)

# -----------------------------
# Load dense embeddings
# -----------------------------
idiom_embeddings = np.load(ARTIFACT_DIR / "idiom_embeddings.npy")

# -----------------------------
# Load index mappings
# -----------------------------
with open(ARTIFACT_DIR / "idiom_to_idx.pkl", "rb") as f:
    idiom_to_idx = pickle.load(f)

with open(ARTIFACT_DIR / "idx_to_idiom.pkl", "rb") as f:
    idx_to_idiom = pickle.load(f)

# -----------------------------
# Load BM25 tokenized bank
# -----------------------------
with open(ARTIFACT_DIR / "bm25_tokens.pkl", "rb") as f:
    idiom_tokens = pickle.load(f)

# -----------------------------
# Load model configuration
# -----------------------------
with open(ARTIFACT_DIR / "model_config.pkl", "rb") as f:
    model_config = pickle.load(f)

print("Artifacts loaded successfully.")
print("Idiom bank size :", len(idiom_bank_df))
print("Embeddings shape:", idiom_embeddings.shape)
print("Dense model     :", model_config["dense_model_name"])
print("Reranker model  :", model_config["reranker_model_name"])

Artifacts loaded successfully.
Idiom bank size : 12522
Embeddings shape: (12522, 384)
Dense model     : sentence-transformers/all-MiniLM-L6-v2
Reranker model  : cross-encoder/ms-marco-MiniLM-L-6-v2


In [27]:
# [B1.4 - FINAL] Rebuild BM25 index from saved tokens

bm25 = BM25Okapi(idiom_tokens)

print("BM25 index rebuilt from saved tokens.")
print("Number of indexed idioms:", len(idiom_tokens))

BM25 index rebuilt from saved tokens.
Number of indexed idioms: 12522


In [28]:
# [B1.5] Load the final embedding model and reranker

# Dense embedder used for semantic retrieval
#embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(model_config["dense_model_name"])

# Cross-encoder used for reranking top candidates
#reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker_model = CrossEncoder(model_config["reranker_model_name"])

print("Embedder and reranker loaded.")

Embedder and reranker loaded.


In [29]:
# [B1.6 - FINAL] Define the Hybrid + Reranker inference pipeline

def minmax_normalize(scores):
    """Normalize scores safely to the [0, 1] range."""
    scores = np.asarray(scores, dtype=float)
    score_min = scores.min()
    score_max = scores.max()

    if score_max - score_min < 1e-12:
        return np.zeros_like(scores, dtype=float)

    return (scores - score_min) / (score_max - score_min)


def predict_idiom(query_text, top_k=3, rerank_top_k=None):
    """
    Run the full Hybrid + Reranker inference pipeline for a single query.

    Returns a pandas DataFrame with:
    - rank
    - idiom_prediction
    - score (normalized reranker score for display)
    """

    # Use saved config by default
    if rerank_top_k is None:
        rerank_top_k = model_config["rerank_top_k"]

    # 1) Encode and normalize query embedding
    query_embedding = embedder.encode(
        [query_text],
        convert_to_numpy=True
    )[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # 2) Dense semantic similarity against full idiom bank
    dense_scores = np.dot(query_embedding, idiom_embeddings.T)

    # 3) BM25 lexical scores against full idiom bank
    bm25_scores = np.asarray(
        bm25.get_scores(query_text.lower().split()),
        dtype=float
    )

    # 4) Normalize dense and lexical scores
    dense_scores_norm = minmax_normalize(dense_scores)
    bm25_scores_norm = minmax_normalize(bm25_scores)

    # 5) Hybrid retrieval using best weights from benchmark
    hybrid_scores = (
        model_config["hybrid_dense_weight"] * dense_scores_norm +
        model_config["hybrid_bm25_weight"] * bm25_scores_norm
    )

    # 6) Keep only top candidates for reranking
    top_candidate_indices = np.argsort(-hybrid_scores)[:rerank_top_k]
    top_candidate_idioms = [idx_to_idiom[idx] for idx in top_candidate_indices]

    # 7) Build query-candidate pairs for reranking
    rerank_pairs = [[query_text, idiom] for idiom in top_candidate_idioms]

    # 8) Score candidates with reranker
    rerank_scores = reranker_model.predict(
        rerank_pairs,
        batch_size=32,
        show_progress_bar=False
    )
    rerank_scores = np.asarray(rerank_scores, dtype=float)

    # 9) Sort by reranker score
    rerank_order = np.argsort(-rerank_scores)
    final_idioms = [top_candidate_idioms[i] for i in rerank_order]
    final_scores = rerank_scores[rerank_order]

    # 10) Normalize final scores for display only
    final_scores_display = minmax_normalize(final_scores)

    # 11) Build result table
    result_df = pd.DataFrame({
        "rank": range(1, len(final_idioms) + 1),
        "idiom_prediction": final_idioms,
        "score": final_scores_display
    })

    # 12) Return only requested top-k
    return result_df.head(top_k)

---

---

In [30]:
# [B1.8] Demo: known idiom-like query

query_text = "She revealed the secret to everyone."

result_df = predict_idiom(query_text, top_k=5)

print(f"Query: {query_text}\n")
result_df

Query: She revealed the secret to everyone.



,rank,idiom_prediction,score
0,1,top secret,1.000000
1,2,best-kept secret,0.941132
2,3,worst-kept secret,0.783825
3,4,tell all,0.630691
4,5,leak out,0.428315


### Multi-Query Demo

We can also test several input sentences in sequence to inspect the model’s behavior across different contexts.

In [31]:
# [B1.9] Demo: multiple queries

demo_queries = [
    "She finally revealed the secret to everyone.",
    "After so many setbacks, he decided to quit.",
    "He avoided answering the question directly.",
]

for query_text in demo_queries:
    print("=" * 100)
    print(f"Query: {query_text}\n")
    display(predict_idiom(query_text, top_k=3))

Query: She finally revealed the secret to everyone.



,rank,idiom_prediction,score
0,1,top secret,1.000000
1,2,worst-kept secret,0.763002
2,3,tell all,0.705245


Query: After so many setbacks, he decided to quit.



,rank,idiom_prediction,score
0,1,back down,1.000000
1,2,three strikes and you're out,0.604444
2,3,down for the count,0.565456


Query: He avoided answering the question directly.



,rank,idiom_prediction,score
0,1,no question,1.000000
1,2,don't ask me,0.538009
2,3,no comment,0.479473


### Demo Note

These examples are intentionally more open-ended than the earlier test cases.

They show that the pipeline is strongest when the context clearly signals a known idiom, while more generic sentences may retrieve semantically related expressions instead of a single exact target.

In [32]:
# [B1.10] Run inference on a custom list of sentences

demo_queries = [
    "He refused to reveal the details and kept everything secret.",
    "After so many failures, she finally gave up.",
    "The negotiations were difficult, but both sides reached an agreement.",
    "He was extremely nervous before the interview.",
    "She said something that made the whole room suddenly silent."
]

for query_text in demo_queries:
    print(f"\nQuery: {query_text}")
    display(predict_idiom(query_text, top_k=3))


Query: He refused to reveal the details and kept everything secret.


,rank,idiom_prediction,score
0,1,worst-kept secret,1.000000
1,2,a magician never reveals his secrets,0.980863
2,3,best-kept secret,0.956234



Query: After so many failures, she finally gave up.


,rank,idiom_prediction,score
0,1,if all else fails,1.000000
1,2,failure is the best teacher,0.823085
2,3,task failed successfully,0.726644



Query: The negotiations were difficult, but both sides reached an agreement.


,rank,idiom_prediction,score
0,1,cut a deal,1.000000
1,2,done deal,0.863466
2,3,more than one bargained for,0.823436



Query: He was extremely nervous before the interview.


,rank,idiom_prediction,score
0,1,nervous hit,1.000000
1,2,on edge,0.546881
2,3,sweat bullets,0.234709



Query: She said something that made the whole room suddenly silent.


,rank,idiom_prediction,score
0,1,deafening silence,1.000000
1,2,dead silence,0.850080
2,3,silence is consent,0.811762


### Conclusion

This notebook demonstrates a lightweight inference pipeline for Task 2: Context → Idiom Retrieval.

The system combines:
- dense semantic retrieval (MiniLM)
- lexical matching (BM25)
- cross-encoder reranking

The final predictions are presented as ranked candidates with normalized confidence scores.

While the model performs strongly when the input clearly matches a known idiom, more general or ambiguous sentences may return semantically related expressions rather than a single exact idiom.

This behavior reflects the inherent difficulty of open-ended idiom retrieval tasks.

debuging

In [58]:
print("idiom_bank_df rows:", len(idiom_bank_df))
print("idiom_embeddings rows:", idiom_embeddings.shape[0])
print("idx_to_idiom size:", len(idx_to_idiom))
print("idiom_to_idx size:", len(idiom_to_idx))
print("bm25 token rows:", len(idiom_tokens))
print("config:", model_config)

idiom_bank_df rows: 12522
idiom_embeddings rows: 12522
idx_to_idiom size: 12522
idiom_to_idx size: 12522
bm25 token rows: 12522
config: {'dense_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'reranker_model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2', 'hybrid_dense_weight': 0.8, 'hybrid_bm25_weight': 0.2, 'rerank_top_k': 10}


In [59]:
def dense_only_debug(query_text, top_k=10):
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    dense_scores = np.dot(query_embedding, idiom_embeddings.T)
    top_indices = np.argsort(-dense_scores)[:top_k]

    rows = []
    for idx in top_indices:
        rows.append({
            "idiom": idx_to_idiom[idx],
            "dense_score": float(dense_scores[idx])
        })

    return pd.DataFrame(rows)


def hybrid_only_debug(query_text, top_k=10):
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    dense_scores = np.dot(query_embedding, idiom_embeddings.T)
    bm25_scores = np.asarray(bm25.get_scores(query_text.lower().split()), dtype=float)

    dense_scores_norm = minmax_normalize(dense_scores)
    bm25_scores_norm = minmax_normalize(bm25_scores)

    hybrid_scores = (
        model_config["hybrid_dense_weight"] * dense_scores_norm +
        model_config["hybrid_bm25_weight"] * bm25_scores_norm
    )

    top_indices = np.argsort(-hybrid_scores)[:top_k]

    rows = []
    for idx in top_indices:
        rows.append({
            "idiom": idx_to_idiom[idx],
            "dense": float(dense_scores_norm[idx]),
            "bm25": float(bm25_scores_norm[idx]),
            "hybrid": float(hybrid_scores[idx])
        })

    return pd.DataFrame(rows)


def reranker_debug(query_text, rerank_top_k=10):
    query_embedding = embedder.encode([query_text], convert_to_numpy=True)[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    dense_scores = np.dot(query_embedding, idiom_embeddings.T)
    bm25_scores = np.asarray(bm25.get_scores(query_text.lower().split()), dtype=float)

    dense_scores_norm = minmax_normalize(dense_scores)
    bm25_scores_norm = minmax_normalize(bm25_scores)

    hybrid_scores = (
        model_config["hybrid_dense_weight"] * dense_scores_norm +
        model_config["hybrid_bm25_weight"] * bm25_scores_norm
    )

    top_candidate_indices = np.argsort(-hybrid_scores)[:rerank_top_k]
    top_candidate_idioms = [idx_to_idiom[idx] for idx in top_candidate_indices]

    rerank_pairs = [[query_text, idiom] for idiom in top_candidate_idioms]
    rerank_scores = reranker_model.predict(
        rerank_pairs,
        batch_size=32,
        show_progress_bar=False
    )
    rerank_scores = np.asarray(rerank_scores, dtype=float)

    rerank_order = np.argsort(-rerank_scores)

    rows = []
    for rank_pos, local_idx in enumerate(rerank_order, start=1):
        bank_idx = top_candidate_indices[local_idx]
        rows.append({
            "rank": rank_pos,
            "idiom": idx_to_idiom[bank_idx],
            "dense": float(dense_scores_norm[bank_idx]),
            "bm25": float(bm25_scores_norm[bank_idx]),
            "hybrid": float(hybrid_scores[bank_idx]),
            "rerank": float(rerank_scores[local_idx])
        })

    return pd.DataFrame(rows)

In [60]:
dense_only_debug("He revealed the secret accidentally.", top_k=10)
hybrid_only_debug("He revealed the secret accidentally.", top_k=10)
reranker_debug("He revealed the secret accidentally.", rerank_top_k=10)

,rank,idiom,dense,bm25,hybrid,rerank
0,1,worst-kept secret,0.933241,0.567996,0.860192,-4.236086
1,2,leak out,1.000000,0.389161,0.877832,-7.838131
2,3,slip out,0.906636,0.811731,0.887655,-8.226645
3,4,let slip,0.959885,0.619496,0.891807,-8.678867
4,5,let something slip,0.983313,0.645469,0.915744,-9.351870
5,6,spill the beans,0.878690,0.482259,0.799404,-9.545348
6,7,give the game away,0.876404,0.626949,0.826513,-9.604462
7,8,murder will out,0.785872,0.781776,0.785053,-9.979750
8,9,pull back the curtain,0.873131,0.424856,0.783476,-10.044349
9,10,the cat's out of the bag,0.825738,1.000000,0.860590,-10.257605
